# El Desafío de la Profundidad en GNNs

**Objetivo:** Identificar los límites arquitectónicos de las GNNs convencionales al incrementar el número de capas.

---

En el módulo anterior exploramos los **fundamentos de las Redes Neuronales de Grafos**: como representar datos relacionales, el paradigma de paso de mensajes y la ecuación general de actualizacion de un nodo. Vimos que cada capa de una GNN agrega información de los vecinos directos.

Surge una pregunta natural: **si una capa captura vecinos a 1 salto, apilar más capas debería capturar relaciones más lejanas, verdad?** En teoría sí, pero en la práctica aparece un problema fundamental que limita la profundidad útil de estas redes.

En este notebook vamos a:
1. **Visualizar** como la agregación repetida suaviza las representaciones de los nodos.
2. **Cuantificar** el fenómeno de *over-smoothing* midiendo distancias entre representaciones.
3. **Analizar** la causa matematica mediante autovalores de la matriz de adyacencia normalizada.
4. **Implementar** conexiones residuales (*skip connections*) y evaluar su efecto.
5. **Plantear** la pregunta de investigacion que nos llevara al siguiente módulo: FractalNets.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import networkx as nx
from scipy import sparse
from scipy.sparse.linalg import eigsh

# Configuracion global de visualización
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 11

# Semilla para reproducibilidad
np.random.seed(42)

## 1. Construcción del grafo de prueba

Vamos a trabajar con el **grafo del Club de Karate de Zachary**, un grafo clásico en ciencia de redes. Tiene 34 nodos y una estructura de comunidades bien definida: dos grupos que se formaron tras la división del club.

Esta estructura es perfecta para nuestro propósito: si asignamos **representaciones iniciales distintas** a cada comunidad, podremos observar como la agregación repetida las mezcla hasta hacerlas indistinguibles.

In [ ]:
# Cargar el grafo del Club de Karate
G = nx.karate_club_graph()
n_nodos = G.number_of_nodes()
n_aristas = G.number_of_edges()

print(f"Grafo del Club de Karate:")
print(f"  Nodos: {n_nodos}")
print(f"  Aristas: {n_aristas}")

# Extraer las etiquetas de comunidad reales
# En networkx, cada nodo tiene un atributo 'club' que indica a que faccion pertenece
comunidades = [0 if G.nodes[i]['club'] == 'Mr. Hi' else 1 for i in range(n_nodos)]
comunidades = np.array(comunidades)

print(f"  Comunidad 0 (Mr. Hi): {np.sum(comunidades == 0)} nodos")
print(f"  Comunidad 1 (Officer): {np.sum(comunidades == 1)} nodos")

# Calcular un layout fijo para todas las visualizaciones
pos = nx.spring_layout(G, seed=42, k=0.5)

# Visualizar el grafo con colores por comunidad
fig, ax = plt.subplots(figsize=(8, 6))
colores_comunidad = ['#e74c3c' if c == 0 else '#3498db' for c in comunidades]
nx.draw_networkx(G, pos, ax=ax, node_color=colores_comunidad, 
                 node_size=300, font_size=8, font_color='white',
                 edge_color='#cccccc', width=0.8)
ax.set_title('Grafo del Club de Karate — Comunidades reales', fontsize=14)

# Leyenda manual
from matplotlib.patches import Patch
leyenda = [Patch(facecolor='#e74c3c', label='Comunidad Mr. Hi'),
           Patch(facecolor='#3498db', label='Comunidad Officer')]
ax.legend(handles=leyenda, loc='upper left', fontsize=10)
plt.tight_layout()
plt.show()

## 2. Asignacion de representaciones iniciales

Para hacer visible el efecto del *over-smoothing*, asignaremos a cada nodo un **vector de representación en $\mathbb{R}^3$** (que interpretaremos como colores RGB). Los nodos de cada comunidad recibiran colores cercanos pero distintos:

- **Comunidad 0 (Mr. Hi):** tonos rojos/calidos
- **Comunidad 1 (Officer):** tonos azules/frios

Además, agregaremos un poco de ruido para que cada nodo tenga una representación ligeramente diferente, incluso dentro de la misma comunidad. Esto simula las **features iniciales heterogeneas** que tendriamos en un problema real.

In [ ]:
# Crear representaciones iniciales en R^3 (interpretables como RGB)
# Comunidad 0: base roja [0.9, 0.2, 0.2]
# Comunidad 1: base azul [0.2, 0.4, 0.9]

H_inicial = np.zeros((n_nodos, 3))

for i in range(n_nodos):
    if comunidades[i] == 0:
        # Tonos rojos/calidos con variación
        H_inicial[i] = [0.9, 0.2, 0.2] + np.random.normal(0, 0.1, 3)
    else:
        # Tonos azules/frios con variación
        H_inicial[i] = [0.2, 0.4, 0.9] + np.random.normal(0, 0.1, 3)

# Asegurar que los valores estan en [0, 1] para interpretarlos como colores
H_inicial = np.clip(H_inicial, 0, 1)

print("Representaciones iniciales (primeros 5 nodos):")
for i in range(5):
    com = "Mr. Hi" if comunidades[i] == 0 else "Officer"
    print(f"  Nodo {i} ({com}): [{H_inicial[i, 0]:.3f}, {H_inicial[i, 1]:.3f}, {H_inicial[i, 2]:.3f}]")

## 3. Construcción de la matriz de adyacencia normalizada $\tilde{A}$

La operación fundamental de una GNN es la **agregación de vecinos**. Matricialmente, esto se expresa como:

$$H^{(k)} = \tilde{A} \cdot H^{(k-1)}$$

donde $\tilde{A}$ es la **matriz de adyacencia normalizada simetricamente con autoconexiones**:

$$\tilde{A} = \hat{D}^{-1/2} \hat{A} \hat{D}^{-1/2}$$

con $\hat{A} = A + I_N$ (adyacencia + autoconexiones) y $\hat{D}$ la matriz diagonal de grados de $\hat{A}$.

Esta normalizacion tiene dos efectos importantes:
1. **Las autoconexiones** ($+I_N$) garantizan que cada nodo conserve parte de su propia información.
2. **La normalizacion por grados** ($\hat{D}^{-1/2}$) evita que los nodos de alto grado dominen la agregación.

> **Nota simplificadora:** En una GNN real, cada capa también aplica una transformación lineal $W^{(k)}$ y una no-linealidad $\sigma$. Aquí omitimos ambas para aislar el efecto puro de la agregación. Esto es equivalente a estudiar el operador de difusión del grafo.

In [ ]:
# Paso 1: Obtener la matriz de adyacencia A
A = nx.adjacency_matrix(G).astype(float)

# Paso 2: Agregar autoconexiones -> A_hat = A + I
I = sparse.eye(n_nodos)
A_hat = A + I

# Paso 3: Calcular la matriz de grados D_hat
grados = np.array(A_hat.sum(axis=1)).flatten()
D_hat_inv_sqrt = sparse.diags(1.0 / np.sqrt(grados))
# Nota: como agregamos I (autoconexiones), el grado mínimo es 1,
# por lo que np.sqrt(grados) nunca es 0 y la división es segura.

# Paso 4: Normalizacion simétrica -> A_tilde = D^{-1/2} A_hat D^{-1/2}
A_tilde = D_hat_inv_sqrt @ A_hat @ D_hat_inv_sqrt

# Convertir a densa para facilitar las operaciones de potencia
A_tilde_dense = A_tilde.toarray()

print(f"Forma de A_tilde: {A_tilde_dense.shape}")
print(f"A_tilde es simétrica: {np.allclose(A_tilde_dense, A_tilde_dense.T)}")
print(f"\nPrimeras 5x5 entradas de A_tilde:")
print(np.round(A_tilde_dense[:5, :5], 4))

## 4. Expansión del campo receptivo

Antes de ver el *over-smoothing*, entendamos **por qué** ocurre. Cada capa de una GNN permite que un nodo "vea" un salto más lejos en el grafo:

| Capas | Alcance del nodo |
|-------|------------------|
| 1     | Vecinos directos (1 salto) |
| 2     | Vecinos de vecinos (2 saltos) |
| $k$   | Todos los nodos a distancia $\leq k$ |

En un grafo pequeño como el Club de Karate (diámetro ~5), con solo **5-6 capas** un nodo ya puede integrar información de **todo** el grafo. Con 10-20 capas, cada nodo ha mezclado la información de todos los demás múltiples veces.

Veamos esto concretamente: para un nodo elegido, calcularemos cuantos nodos alcanza en función del número de saltos.

In [ ]:
# Elegimos el nodo 0 (un nodo central en la comunidad Mr. Hi)
nodo_ejemplo = 0

# Calcular el campo receptivo en función del número de saltos
alcance = []
max_saltos = 8

for k in range(max_saltos + 1):
    # Nodos alcanzables en exactamente k saltos o menos
    if k == 0:
        nodos_alcanzados = {nodo_ejemplo}
    else:
        nodos_alcanzados = set()
        for nodo in nx.single_source_shortest_path_length(G, nodo_ejemplo, cutoff=k):
            nodos_alcanzados.add(nodo)
    alcance.append(len(nodos_alcanzados))

# Visualizar la expansión del campo receptivo
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
saltos_a_mostrar = [1, 2, 3, 5]

for idx, k in enumerate(saltos_a_mostrar):
    ax = axes[idx]
    # Calcular nodos alcanzados en k saltos
    nodos_k = set(nx.single_source_shortest_path_length(G, nodo_ejemplo, cutoff=k).keys())
    
    # Colores: rojo = nodo fuente, naranja = alcanzados, gris = no alcanzados
    colores = []
    for i in range(n_nodos):
        if i == nodo_ejemplo:
            colores.append('#e74c3c')  # Rojo: nodo fuente
        elif i in nodos_k:
            colores.append('#f39c12')  # Naranja: alcanzado
        else:
            colores.append('#bdc3c7')  # Gris: no alcanzado
    
    nx.draw_networkx(G, pos, ax=ax, node_color=colores, node_size=200,
                     font_size=6, font_color='white', edge_color='#dddddd', width=0.5)
    ax.set_title(f'{k} salto{"s" if k > 1 else ""}: {len(nodos_k)}/{n_nodos} nodos', fontsize=12)

plt.suptitle(f'Expansión del campo receptivo del nodo {nodo_ejemplo}', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# Grafica cuantitativa
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(range(max_saltos + 1), alcance, color='#3498db', alpha=0.8, edgecolor='#2c3e50')
ax.axhline(y=n_nodos, color='#e74c3c', linestyle='--', label=f'Total de nodos ({n_nodos})')
ax.set_xlabel('Numero de saltos (= capas de GNN)', fontsize=12)
ax.set_ylabel('Nodos alcanzados', fontsize=12)
ax.set_title(f'Campo receptivo del nodo {nodo_ejemplo} vs. profundidad', fontsize=13)
ax.legend(fontsize=11)
ax.set_xticks(range(max_saltos + 1))
plt.tight_layout()
plt.show()

print(f"Diametro del grafo: {nx.diameter(G)}")
print(f"Con {nx.diameter(G)} capas, el nodo {nodo_ejemplo} ya 've' a TODOS los nodos del grafo.")

## 5. Visualización del Over-smoothing

Ahora viene el experimento central. Vamos a aplicar la agregación repetida:

$$H^{(k)} = \tilde{A} \cdot H^{(k-1)}$$

partiendo de las representaciones iniciales $H^{(0)}$ (colores distintos por comunidad) y visualizando como evolucionan los colores de los nodos tras 1, 2, 5, 10 y 20 rondas de agregación.

**Prediccion:** si el *over-smoothing* es real, los colores deberian converger a un solo tono uniforme, perdiendo toda la información de comunidad.

In [ ]:
def aplicar_agregacion(A_tilde, H_0, num_capas):
    """Aplica la agregación k veces: H^(k) = A_tilde * H^(k-1)."""
    H = H_0.copy()
    for _ in range(num_capas):
        H = A_tilde @ H
    return H

# Numero de capas a visualizar
capas_a_mostrar = [0, 1, 2, 5, 10, 20]

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

# Guardar las representaciones para análisis posterior
representaciones = {}

for idx, k in enumerate(capas_a_mostrar):
    ax = axes[idx]
    
    # Calcular H^(k)
    H_k = aplicar_agregacion(A_tilde_dense, H_inicial, k)
    representaciones[k] = H_k.copy()
    
    # Normalizar a [0,1] para visualización como color
    H_vis = H_k.copy()
    # Normalizacion min-max por columna para preservar la variabilidad relativa
    for col in range(3):
        col_min = H_vis[:, col].min()
        col_max = H_vis[:, col].max()
        if col_max - col_min > 1e-10:
            H_vis[:, col] = (H_vis[:, col] - col_min) / (col_max - col_min)
        else:
            H_vis[:, col] = 0.5  # Si todos son iguales, gris medio
    
    # Usar los valores RGB como colores de nodo
    colores_nodos = [H_vis[i] for i in range(n_nodos)]
    
    nx.draw_networkx(G, pos, ax=ax, node_color=colores_nodos,
                     node_size=300, font_size=7, font_color='white',
                     edge_color='#dddddd', width=0.5)
    
    # Calcular la desviación estandar de las representaciones como medida de diversidad
    diversidad = np.mean(np.std(H_k, axis=0))
    ax.set_title(f'k = {k} capas (diversidad: {diversidad:.4f})', fontsize=12)

plt.suptitle('Evolucion de las representaciones con la profundidad\n'
             '(los colores se vuelven uniformes = over-smoothing)',
             fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

Observa como los nodos, que inicialmente tenian colores claramente distintos entre comunidades, **convergen a representaciones alineadas** a medida que aumentamos el número de capas. Con $k=20$, la información de comunidad original se ha diluido significativamente.

Esto es exactamente el **over-smoothing**: la agregación repetida actua como un filtro de paso bajo que elimina las diferencias locales. Las representaciones no se vuelven estrictamente *idénticas*, sino **colineales** (apuntan en la misma dirección, difiriendo solo por un factor de escala segun el nodo), lo que las hace poco discriminativas para clasificación.

## 6. Cuantificacion del Over-smoothing

La visualización es convincente, pero necesitamos una **metrica cuantitativa**. Vamos a medir:

1. **Distancia promedio entre representaciones de nodos de diferentes comunidades** (inter-clase): queremos que sea alta (nodos de distintas comunidades deberian verse diferentes).
2. **Distancia promedio entre representaciones de nodos de la misma comunidad** (intra-clase): queremos que sea baja (nodos de la misma comunidad deberian verse similares).

Si el *over-smoothing* ocurre, **ambas distancias convergen a cero**, porque todos los nodos se vuelven iguales.

In [ ]:
from scipy.spatial.distance import pdist, squareform

def calcular_distancias(H, comunidades):
    """Calcula las distancias promedio intra-clase e inter-clase."""
    # Matriz de distancias euclideas entre todos los pares
    dist_matrix = squareform(pdist(H, metric='euclidean'))
    
    # Separar pares intra-clase e inter-clase
    intra_dists = []
    inter_dists = []
    
    for i in range(len(comunidades)):
        for j in range(i + 1, len(comunidades)):
            if comunidades[i] == comunidades[j]:
                intra_dists.append(dist_matrix[i, j])
            else:
                inter_dists.append(dist_matrix[i, j])
    
    return np.mean(intra_dists), np.mean(inter_dists)

# Evaluar para un rango amplio de profundidades
profundidades = list(range(0, 31))
dist_intra = []
dist_inter = []
diversidad_global = []

for k in profundidades:
    H_k = aplicar_agregacion(A_tilde_dense, H_inicial, k)
    intra, inter = calcular_distancias(H_k, comunidades)
    dist_intra.append(intra)
    dist_inter.append(inter)
    # Diversidad global: desviación estandar promedio de las representaciones
    diversidad_global.append(np.mean(np.std(H_k, axis=0)))

# Grafica
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

# Panel izquierdo: distancias intra e inter clase
ax1.plot(profundidades, dist_inter, 'o-', color='#e74c3c', label='Inter-clase (entre comunidades)', linewidth=2, markersize=4)
ax1.plot(profundidades, dist_intra, 's-', color='#3498db', label='Intra-clase (misma comunidad)', linewidth=2, markersize=4)
ax1.set_xlabel('Numero de capas (k)', fontsize=12)
ax1.set_ylabel('Distancia euclidea promedio', fontsize=12)
ax1.set_title('Distancias entre representaciones vs. profundidad', fontsize=13)
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)
ax1.set_yscale('log')  # Escala logaritmica para ver la convergencia

# Panel derecho: diversidad global
ax2.plot(profundidades, diversidad_global, 'D-', color='#2ecc71', linewidth=2, markersize=4)
ax2.set_xlabel('Numero de capas (k)', fontsize=12)
ax2.set_ylabel('Diversidad (desv. estandar promedio)', fontsize=12)
ax2.set_title('Diversidad de representaciones vs. profundidad', fontsize=13)
ax2.grid(True, alpha=0.3)
ax2.set_yscale('log')

plt.tight_layout()
plt.show()

print(f"Distancia inter-clase inicial (k=0): {dist_inter[0]:.4f}")
print(f"Distancia inter-clase final  (k=30): {dist_inter[-1]:.6f}")
print(f"\nReduccion: {dist_inter[0]/max(dist_inter[-1], 1e-15):.0f}x")
print(f"\n--> Las representaciones de comunidades distintas se han vuelto mucho menos separables.")

Los resultados son claros:
- La **distancia inter-clase** (rojo) cae drasticamente, indicando que nodos de comunidades distintas se vuelven poco distinguibles.
- La **distancia intra-clase** (azul) también cae, porque las representaciones se alinean progresivamente.
- La **diversidad global** decrece hasta valores muy bajos.

**Nota:** Las distancias no llegan exactamente a cero porque las representaciones convergen a escalados del autovector dominante $\mathbf{u}_1$, no a un único punto. Cada nodo retiene un factor de escala distinto ($u_1(v)$), lo que produce distancia euclidiana residual. Sin embargo, la *dirección* de todos los vectores converge, haciendo que la similitud coseno entre nodos tienda a 1 y la clasificación sea imposible.

## 7. Análisis espectral: por qué ocurre el Over-smoothing

La razon matematica del *over-smoothing* esta en las **propiedades espectrales** de la matriz $\tilde{A}$.

### Intuicion

Toda matriz simétrica $\tilde{A}$ se puede descomponer como:

$$\tilde{A} = \sum_{i=1}^{n} \lambda_i \, \mathbf{u}_i \mathbf{u}_i^T$$

donde $\lambda_1 \geq \lambda_2 \geq \ldots \geq \lambda_n$ son los autovalores y $\mathbf{u}_i$ los autovectores correspondientes.

Cuando elevamos $\tilde{A}$ a la potencia $k$:

$$\tilde{A}^k = \sum_{i=1}^{n} \lambda_i^k \, \mathbf{u}_i \mathbf{u}_i^T$$

Si $|\lambda_1| > |\lambda_i|$ para todo $i > 1$ (es decir, el autovalor dominante es estrictamente mayor en magnitud), entonces cuando $k \to \infty$:

$$\tilde{A}^k \approx \lambda_1^k \, \mathbf{u}_1 \mathbf{u}_1^T$$

Esto significa que **todas las filas de $\tilde{A}^k$ se alinean con $\mathbf{u}_1$**, el autovector principal. Consecuencia: $\tilde{A}^k H$ produce representaciones proporcionales a $\mathbf{u}_1$, es decir, **todos los nodos obtienen representaciones alineadas con $\mathbf{u}_1$**, diferenciándose solo por un factor escalar $u_1(v)$ que depende del nodo. Esto las hace prácticamente indistinguibles para tareas de clasificación, aunque no sean estrictamente idénticas.

In [ ]:
# Calcular autovalores y autovectores de A_tilde
autovalores, autovectores = np.linalg.eigh(A_tilde_dense)

# Ordenar de mayor a menor (eigh devuelve en orden ascendente)
idx_orden = np.argsort(-np.abs(autovalores))
autovalores = autovalores[idx_orden]
autovectores = autovectores[:, idx_orden]

print("Los 10 autovalores más grandes (en magnitud) de A_tilde:")
for i in range(10):
    print(f"  lambda_{i+1} = {autovalores[i]:.6f}")

print(f"\nAutovalor dominante: lambda_1 = {autovalores[0]:.6f}")
print(f"Segundo autovalor:   lambda_2 = {autovalores[1]:.6f}")
print(f"\nBrecha espectral (lambda_1 - |lambda_2|): {autovalores[0] - abs(autovalores[1]):.6f}")
print(f"Ratio |lambda_2/lambda_1|: {abs(autovalores[1]/autovalores[0]):.6f}")
print(f"\nEste ratio < 1 garantiza que lambda_2^k -> 0 cuando k -> infinito,")
print(f"dejando solo la contribucion del autovector principal.")

In [ ]:
# Visualizar la convergencia de A_tilde^k hacia la proyeccion sobre el autovector principal

# Autovector dominante (u_1)
u1 = autovectores[:, 0].reshape(-1, 1)

# Matriz de proyeccion: u_1 * u_1^T (multiplicada por lambda_1^k)
# Cómo lambda_1 = 1 para la normalizada con autoconexiones, converge a u_1 * u_1^T

# Calcular el error de aproximacion para distintas profundidades
profundidades_esp = list(range(0, 51))
errores_aprox = []

# Proyeccion dominante: lambda_1^k * u_1 * u_1^T
for k in profundidades_esp:
    A_k = np.linalg.matrix_power(A_tilde_dense, k)
    A_k_aprox = (autovalores[0] ** k) * (u1 @ u1.T)
    error = np.linalg.norm(A_k - A_k_aprox, 'fro') / np.linalg.norm(A_k, 'fro') if np.linalg.norm(A_k, 'fro') > 1e-15 else 0
    errores_aprox.append(error)

# Grafica del decaimiento de autovalores elevados a potencia k
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

# Panel izquierdo: autovalores elevados a la k
k_vals = np.arange(0, 31)
for i in range(5):
    ax1.plot(k_vals, np.abs(autovalores[i]) ** k_vals, 
             label=f'|$\\lambda_{{{i+1}}}$|$^k$ = |{autovalores[i]:.3f}|$^k$',
             linewidth=2)

ax1.set_xlabel('Profundidad k (capas)', fontsize=12)
ax1.set_ylabel('$|\\lambda_i|^k$', fontsize=12)
ax1.set_title('Decaimiento de las contribuciones espectrales', fontsize=13)
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)

# Panel derecho: error de la aproximacion de rango 1
ax2.plot(profundidades_esp, errores_aprox, 'o-', color='#9b59b6', linewidth=2, markersize=3)
ax2.set_xlabel('Profundidad k (capas)', fontsize=12)
ax2.set_ylabel('Error relativo de la aprox. rango 1', fontsize=12)
ax2.set_title('$\\tilde{A}^k$ converge a $\\lambda_1^k \\mathbf{u}_1 \\mathbf{u}_1^T$', fontsize=13)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Con k=30, el error de la aproximacion de rango 1 es: {errores_aprox[30]:.6f}")
print(f"Con k=50, el error de la aproximacion de rango 1 es: {errores_aprox[50]:.6f}")
print(f"\n--> A_tilde^k esta casi completamente determinada por un solo autovector.")
print(f"    Las representaciones de los nodos se alinean progresivamente con u_1, perdiendo discriminabilidad.")

In [ ]:
# Visualizar el autovector dominante u_1
# Este es el vector al que convergen TODAS las representaciones

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

# Panel izquierdo: valores del autovector dominante por nodo
colores_barra = ['#e74c3c' if c == 0 else '#3498db' for c in comunidades]
ax1.bar(range(n_nodos), u1.flatten(), color=colores_barra, edgecolor='#2c3e50', linewidth=0.5)
ax1.set_xlabel('Indice del nodo', fontsize=12)
ax1.set_ylabel('Valor en $\\mathbf{u}_1$', fontsize=12)
ax1.set_title('Autovector dominante $\\mathbf{u}_1$ de $\\tilde{A}$', fontsize=13)
ax1.axhline(y=0, color='black', linewidth=0.5)

# Panel derecho: el autovector sobre el grafo
valores_u1 = u1.flatten()
nx.draw_networkx(G, pos, ax=ax2, node_color=valores_u1, cmap='RdYlBu_r',
                 node_size=300, font_size=7, font_color='white',
                 edge_color='#dddddd', width=0.5)
sm = plt.cm.ScalarMappable(cmap='RdYlBu_r', 
                           norm=plt.Normalize(vmin=valores_u1.min(), vmax=valores_u1.max()))
sm.set_array([])
plt.colorbar(sm, ax=ax2, label='Valor en $\\mathbf{u}_1$')
ax2.set_title('$\\mathbf{u}_1$ visualizado sobre el grafo', fontsize=13)

plt.tight_layout()
plt.show()

print("Observa que u_1 tiene valores similares para todos los nodos (todos positivos).")
print("Esto confirma que, al alinearse con u_1, las representaciones pierden discriminabilidad.")
print(f"\nRango de u_1: [{valores_u1.min():.4f}, {valores_u1.max():.4f}]")
print(f"Desviacion estandar de u_1: {np.std(valores_u1):.4f}")

### Resumen del análisis espectral

El análisis confirma la teoría:

1. El autovalor dominante $\lambda_1 \approx 1$ **no decae** con la profundidad.
2. Todos los demas autovalores $|\lambda_i| < 1$ **decaen exponencialmente** (son elevados a la potencia $k$).
3. Despues de suficientes capas, $\tilde{A}^k \approx \lambda_1^k \mathbf{u}_1 \mathbf{u}_1^T$.
4. El autovector dominante $\mathbf{u}_1$ es suave (todos los valores son positivos y similares), por lo que las representaciones resultantes pierden toda discriminabilidad.

Este es el mecanismo fundamental del **suavizado laplaciano** que causa el *over-smoothing*.

---

## 8. Solucion tradicional: Conexiones residuales (Skip Connections)

La solución clásica, tomada de las **ResNets** (He et al., 2016) para redes convolucionales, es agregar una **conexión residual** que suma la entrada de la capa a su salida.

En la práctica, se usa un parámetro $\alpha \in [0, 1]$ para controlar el balance entre la agregación de vecinos y la preservación de la propia representación:

$$H^{(k)} = (1 - \alpha) \cdot \tilde{A} \cdot H^{(k-1)} + \alpha \cdot H^{(k-1)}$$

que se puede reescribir como:

$$H^{(k)} = \left[(1 - \alpha) \tilde{A} + \alpha I\right] \cdot H^{(k-1)}$$

donde $\alpha = 0$ es la GNN pura (sin residuales) y $\alpha = 1$ ignora completamente la agregación.

La idea es sencilla pero poderosa: al incluir el término $\alpha \cdot H^{(k-1)}$, **anclamos** las representaciones a sus valores anteriores, resistiendo la fuerza de difusión de la agregación.

Implementemos esto y veamos como afecta al *over-smoothing*.

In [ ]:
def aplicar_agregacion_residual(A_tilde, H_0, num_capas, alpha=0.2):
    """Aplica agregación con conexión residual.
    
    H^(k) = (1 - alpha) * A_tilde * H^(k-1) + alpha * H^(k-1)
    
    Parametros:
        A_tilde: Matriz de adyacencia normalizada
        H_0: Representaciones iniciales
        num_capas: Numero de rondas de agregación
        alpha: Peso de la conexión residual (0 = sin residual, 1 = sin agregación)
    """
    H = H_0.copy()
    for _ in range(num_capas):
        H_agg = A_tilde @ H          # Agregacion de vecinos
        H = (1 - alpha) * H_agg + alpha * H  # Mezcla con residual
    return H

# Comparar varios valores de alpha
alphas = [0.0, 0.1, 0.2, 0.5]
profundidades_res = list(range(0, 51))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))
colores_alpha = ['#e74c3c', '#f39c12', '#2ecc71', '#3498db']

for i, alpha in enumerate(alphas):
    dist_inter_res = []
    diversidad_res = []
    
    for k in profundidades_res:
        H_k = aplicar_agregacion_residual(A_tilde_dense, H_inicial, k, alpha)
        _, inter = calcular_distancias(H_k, comunidades)
        dist_inter_res.append(inter)
        diversidad_res.append(np.mean(np.std(H_k, axis=0)))
    
    etiqueta = f'alpha={alpha}' + (' (sin residual)' if alpha == 0 else '')
    ax1.plot(profundidades_res, dist_inter_res, '-', color=colores_alpha[i],
             label=etiqueta, linewidth=2)
    ax2.plot(profundidades_res, diversidad_res, '-', color=colores_alpha[i],
             label=etiqueta, linewidth=2)

ax1.set_xlabel('Numero de capas (k)', fontsize=12)
ax1.set_ylabel('Distancia inter-clase', fontsize=12)
ax1.set_title('Efecto de skip connections en la separabilidad', fontsize=13)
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)
ax1.set_yscale('log')

ax2.set_xlabel('Numero de capas (k)', fontsize=12)
ax2.set_ylabel('Diversidad', fontsize=12)
ax2.set_title('Efecto de skip connections en la diversidad', fontsize=13)
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)
ax2.set_yscale('log')

plt.tight_layout()
plt.show()

In [ ]:
# Visualización lado a lado: sin residual vs. con residual (alpha=0.2) a k=20 capas

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

configs = [
    ('Inicial (k=0)', H_inicial),
    ('Sin residual (k=20)', aplicar_agregacion(A_tilde_dense, H_inicial, 20)),
    ('Con residual alpha=0.2 (k=20)', aplicar_agregacion_residual(A_tilde_dense, H_inicial, 20, alpha=0.2)),
]

for idx, (titulo, H_k) in enumerate(configs):
    ax = axes[idx]
    
    # Normalizar para visualización
    H_vis = H_k.copy()
    for col in range(3):
        col_min = H_vis[:, col].min()
        col_max = H_vis[:, col].max()
        if col_max - col_min > 1e-10:
            H_vis[:, col] = (H_vis[:, col] - col_min) / (col_max - col_min)
        else:
            H_vis[:, col] = 0.5
    
    colores_nodos = [H_vis[i] for i in range(n_nodos)]
    nx.draw_networkx(G, pos, ax=ax, node_color=colores_nodos,
                     node_size=300, font_size=7, font_color='white',
                     edge_color='#dddddd', width=0.5)
    
    diversidad = np.mean(np.std(H_k, axis=0))
    ax.set_title(f'{titulo}\n(diversidad: {diversidad:.4f})', fontsize=11)

plt.suptitle('Comparacion: efecto de las conexiones residuales a 20 capas', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

### Análisis de las conexiones residuales

Los resultados muestran que las conexiones residuales **mitigan** el *over-smoothing* pero **no lo resuelven por completo**:

- Con $\alpha > 0$, la convergencia se ralentiza significativamente.
- Las representaciones retienen más diversidad a profundidades moderadas.
- Sin embargo, para profundidades muy grandes, el suavizado sigue ocurriendo (aunque a una tasa menor).

El mecanismo residual funciona porque **ancla** cada representación a su valor de la capa anterior, resistiendo la fuerza de difusión de la agregación. Pero estructuralmente, sigue siendo una solución aditiva: estamos sumando una señal de identidad a la salida de la capa.

**Limite teórico:** En la práctica, las GNNs modernas con conexiones residuales (GraphSAGE, GCN + skip) también sufren de este límite. Si $k \to \infty$, el *over-smoothing* es inevitable en cualquier arquitectura basada puramente en difusión/agregación, porque la operación de promediado de vecinos es inherentemente un filtro pasa-bajas. Esto justifica buscar **alternativas topológicas** que no dependan únicamente de apilar capas de agregación en serie.

Esto nos lleva a una pregunta fundamental...

In [ ]:
# Análisis espectral de la agregación residual:
# La matriz efectiva es M = (1-alpha)*A_tilde + alpha*I
# Sus autovalores son: mu_i = (1-alpha)*lambda_i + alpha

alpha_ejemplo = 0.2
autovalores_residual = (1 - alpha_ejemplo) * autovalores + alpha_ejemplo

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

# Panel izquierdo: espectro original vs. residual
x = np.arange(len(autovalores))
ax1.bar(x - 0.2, np.abs(autovalores), width=0.4, color='#e74c3c', alpha=0.7, label='Sin residual ($\\lambda_i$)')
ax1.bar(x + 0.2, np.abs(autovalores_residual), width=0.4, color='#3498db', alpha=0.7, 
        label=f'Con residual ($\\mu_i$, alpha={alpha_ejemplo})')
ax1.set_xlabel('Indice del autovalor', fontsize=12)
ax1.set_ylabel('|Autovalor|', fontsize=12)
ax1.set_title('Espectro: original vs. con residual', fontsize=13)
ax1.legend(fontsize=10)
ax1.set_xlim(-1, 15)  # Mostrar solo los primeros

# Panel derecho: brecha espectral
alphas_analisis = np.linspace(0, 0.5, 50)
brechas = []
ratios = []
for a in alphas_analisis:
    mu = (1 - a) * autovalores + a
    brecha = np.abs(mu[0]) - np.abs(mu[1])
    ratio = np.abs(mu[1]) / np.abs(mu[0]) if np.abs(mu[0]) > 0 else 0
    brechas.append(brecha)
    ratios.append(ratio)

ax2.plot(alphas_analisis, ratios, '-', color='#9b59b6', linewidth=2)
ax2.set_xlabel('alpha (peso residual)', fontsize=12)
ax2.set_ylabel('$|\\mu_2 / \\mu_1|$', fontsize=12)
ax2.set_title('Ratio del segundo autovalor vs. alpha\n(más cercano a 1 = convergencia más lenta)', fontsize=12)
ax2.grid(True, alpha=0.3)
ax2.axhline(y=1, color='#e74c3c', linestyle='--', alpha=0.5, label='Limite (sin convergencia)')
ax2.legend(fontsize=10)

plt.tight_layout()
plt.show()

print(f"Sin residual: |lambda_2/lambda_1| = {abs(autovalores[1]/autovalores[0]):.4f}")
print(f"Con residual (alpha={alpha_ejemplo}): |mu_2/mu_1| = {abs(autovalores_residual[1]/autovalores_residual[0]):.4f}")
print(f"\nLa conexión residual acerca el ratio a 1, ralentizando la convergencia.")
print(f"Pero mientras el ratio sea < 1, el over-smoothing sigue ocurriendo eventualmente.")

El análisis espectral confirma lo que observamos:

- La conexión residual transforma los autovalores de $\lambda_i$ a $\mu_i = (1-\alpha)\lambda_i + \alpha$.
- Esto **comprime** el espectro hacia 1, haciendo que $|\mu_2/\mu_1|$ sea más cercano a 1.
- Resultado: la convergencia al autovector dominante es **más lenta**, pero no se elimina.
- La única forma de evitar completamente la convergencia sería $|\mu_2/\mu_1| = 1$, lo cual requeriria $\alpha = 1$ (es decir, no agregar nada de información de vecinos, anulando el propósito de la GNN).

---

## Resumen del Módulo 2

En este notebook hemos establecido tres hechos fundamentales:

| Concepto | Observacion |
|----------|-------------|
| **Campo receptivo** | El radio en hops crece linealmente con la profundidad (1 capa = 1 salto). Sin embargo, el *número de nodos alcanzados* puede crecer muy rápido en grafos densos; con ~5 capas ya se cubre todo un grafo pequeño. |
| **Over-smoothing** | La agregación repetida ($\tilde{A}^k H$) hace converger todas las representaciones al autovector dominante $\mathbf{u}_1$. |
| **Conexiones residuales** | Mitigan el problema (ralentizan la convergencia) pero no lo resuelven fundamentalmente. |

Las conexiones residuales, inspiradas en las **ResNets**, son la solución estandar. Pero su mecanismo es una **suma aditiva de identidad**: simplemente anclan la representación a su valor anterior.

---

## Pregunta de transición: Hacia el Módulo 3

### Es posible diseñar una red ultra-profunda que evite la degradacion sin utilizar sumas residuales de identidad?

Las ResNets resuelven el problema de la profundidad sumando un **atajo de identidad** a la salida de cada capa. Pero, qué pasaría si en lugar de sumar atajos, diseñáramos una **topología completamente diferente** donde coexistan rutas de distinta profundidad de forma natural?

En el siguiente módulo exploraremos las **FractalNets** (Larsson et al., 2017), una arquitectura que logra profundidad efectiva mediante **autosimilitud estructural** en lugar de conexiones residuales.

> **Nota importante:** FractalNet fue desarrollada originalmente para **CNNs** (redes convolucionales en visión por computadora), **no para GNNs**. Sin embargo, el problema de fondo es el mismo: la dificultad de entrenar redes muy profundas sin que las representaciones se degraden. La solución estructural que proponen es **conceptualmente transferible** al dominio de grafos, y esa conexión será el tema central de los módulos finales.

**Para reflexionar:**
- En las ResNets, la identidad tiene un estatus *privilegiado* (es la señal que se preserva). Es posible una arquitectura donde **ninguna ruta tenga estatus especial**?
- Si tuvieramos multiples caminos de diferente profundidad operando en paralelo, como deberiamos combinar sus salidas?